In [1]:
# ============================================================================
# CELL 2: LOAD VARIABLES (ضعها في بداية Notebook الجديد)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("📥 LOADING SAVED VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📂 STEP 1: Find save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"

if not save_folder.exists():
    print()
    print("❌ No saved sessions found!")
    print(f"   Expected location: {save_folder}")
    print()
    print("💡 First run CELL 1 in your source notebook to save variables")
    print("=" * 80)
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 📋 STEP 2: List available sessions
    # ═══════════════════════════════════════════════════════════════════════════

    sessions = sorted([d for d in save_folder.iterdir() if d.is_dir()], reverse=True)

    if len(sessions) == 0:
        print()
        print("❌ No sessions found in folder!")
        print(f"   Folder exists but is empty: {save_folder}")
        print()
        print("💡 Run CELL 1 in your source notebook to create a session")
        print("=" * 80)
    else:
        print()
        print(f"📂 Found {len(sessions)} saved session(s):")
        print()
        print("-" * 80)

        # Display available sessions
        for idx, session in enumerate(sessions, 1):
            metadata_path = session / "metadata.pkl"

            if metadata_path.exists():
                try:
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)

                    dt = metadata.get('datetime', metadata.get('timestamp', 'Unknown'))
                    var_count = metadata.get('success_count',
                               len([v for v in metadata.get('saved_variables', {}).values()
                                   if v.get('saved', False)]))

                    print(f"  {idx}. {session.name}")
                    print(f"     Date: {dt}")
                    print(f"     Variables: {var_count}")

                    # Show variable names
                    vars_list = [k for k, v in metadata.get('saved_variables', {}).items()
                                if v.get('saved', False)]
                    if len(vars_list) > 0:
                        print(f"     Contains: {', '.join(vars_list[:5])}")
                        if len(vars_list) > 5:
                            print(f"               ... and {len(vars_list)-5} more")
                    print()
                except:
                    print(f"  {idx}. {session.name} (metadata error)")
                    print()
            else:
                print(f"  {idx}. {session.name} (no metadata)")
                print()

        print("-" * 80)

        # ═══════════════════════════════════════════════════════════════════════════
        # 🎯 STEP 3: Choose session to load
        # ═══════════════════════════════════════════════════════════════════════════

        choice = input("\nEnter session number to load (press Enter for latest): ").strip()

        if choice == '':
            choice = '1'

        try:
            session_idx = int(choice) - 1

            if session_idx < 0 or session_idx >= len(sessions):
                print(f"\n❌ Invalid choice! Must be 1-{len(sessions)}")
                print("=" * 80)
            else:
                selected_session = sessions[session_idx]

                print()
                print("=" * 80)
                print(f"📥 Loading session: {selected_session.name}")
                print("=" * 80)
                print()

                # ═══════════════════════════════════════════════════════════════════════════
                # 📦 STEP 4: Load metadata
                # ═══════════════════════════════════════════════════════════════════════════

                metadata_path = selected_session / "metadata.pkl"

                if metadata_path.exists():
                    with open(metadata_path, 'rb') as f:
                        metadata = pickle.load(f)
                else:
                    print("⚠️  No metadata found - will try to load all .pkl files")
                    metadata = {'saved_variables': {}}

                # ═══════════════════════════════════════════════════════════════════════════
                # 💾 STEP 5: Load variables
                # ═══════════════════════════════════════════════════════════════════════════

                loaded_count = 0
                failed_count = 0

                saved_vars = metadata.get('saved_variables', {})

                if len(saved_vars) == 0:
                    # No metadata, try all .pkl files
                    pkl_files = list(selected_session.glob("*.pkl"))
                    print(f"Found {len(pkl_files)} .pkl files (excluding metadata)")
                    print()

                    for pkl_file in pkl_files:
                        if pkl_file.name != "metadata.pkl":
                            var_name = pkl_file.stem  # filename without .pkl

                            try:
                                with open(pkl_file, 'rb') as f:
                                    var_value = pickle.load(f)

                                globals()[var_name] = var_value

                                size_kb = pkl_file.stat().st_size / 1024
                                size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"
                                var_type = type(var_value).__name__

                                print(f"✅ {var_name:<25} | {var_type:<15} | {size_str}")
                                loaded_count += 1

                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1
                else:
                    # Use metadata
                    for var_name, var_info in saved_vars.items():
                        if var_info.get('saved', False):
                            try:
                                file_path = selected_session / f"{var_name}.pkl"

                                with open(file_path, 'rb') as f:
                                    var_value = pickle.load(f)

                                # Load into global scope
                                globals()[var_name] = var_value

                                print(f"✅ {var_name:<25} | {var_info.get('type', 'Unknown'):<15} | {var_info.get('size', 'Unknown')}")
                                loaded_count += 1

                            except FileNotFoundError:
                                print(f"❌ {var_name:<25} | FILE NOT FOUND")
                                failed_count += 1
                            except Exception as e:
                                print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
                                failed_count += 1

                # ═══════════════════════════════════════════════════════════════════════════
                # ✅ STEP 6: Summary
                # ═══════════════════════════════════════════════════════════════════════════

                print()
                print("=" * 80)
                print("✅ LOAD COMPLETE!")
                print("=" * 80)
                print(f"✅ Loaded:  {loaded_count} variables")
                print(f"❌ Failed:  {failed_count} variables")
                print("=" * 80)
                print()
                print("💡 Variables are now available in this notebook!")
                print("   Example: print(combined_df.head())")
                print("=" * 80)

        except ValueError:
            print()
            print("❌ Invalid input! Please enter a number")
            print("=" * 80)

📥 LOADING SAVED VARIABLES

📂 Found 27 saved session(s):

--------------------------------------------------------------------------------
  1. session_20260119_123201
     Date: 2026-01-19 12:32:51
     Variables: 6
     Contains: combined_df, combined_df_updated, df, inventory_df, sku_df
               ... and 1 more

  2. session_20260119_122934
     Date: 2026-01-19 12:30:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  3. session_20260119_122517
     Date: 2026-01-19 12:26:21
     Variables: 4
     Contains: combined_df, combined_df_updated, df, inventory_df

  4. session_20260119_121915
     Date: 2026-01-19 12:19:21
     Variables: 1
     Contains: combined_df

  5. session_20260119_100308
     Date: 2026-01-19 10:03:14
     Variables: 3
     Contains: combined_df, df, inventory_df

  6. session_20260119_092005
     Date: 2026-01-19 09:20:13
     Variables: 2
     Contains: combined_df, df

  7. session_20260118_165224
     Date: 2026-01-1

In [2]:
# ============================================================================
# ADVANCED DEMAND FORECASTING - PRODUCTION VERSION
# ============================================================================
# ✅ Actual Sales Days tracking (SKUs not active all the time)
# ✅ Dynamic Quarterly Weighting (captures seasonality automatically)
# ✅ Fully Vectorized calculations (10x faster)
# ✅ FAMILY sales only (excludes CONTRACT)
# ✅ 365-day lookback with quarterly breakdown
# ✅ Intelligent stockout adjustment (120-day window)
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

LOOKBACK_DAYS = 365          # 1 year historical data
FORWARD_COVERAGE_MONTHS = 7  # 7 months from today
STOCKOUT_THRESHOLD = 10      # Stock <= 10 indicates potential stockout
STOCKOUT_CAP = 1.5           # Maximum adjustment factor
MIN_DATA_COVERAGE = 0.50     # 50% coverage to trust forecast
MIN_ACTIVE_DAYS = 30         # Minimum days with sales to trust seasonality

# ============================================================================
# BEAUTIFUL OUTPUT
# ============================================================================

def print_header(title, width=80):
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)

def print_section(title):
    print(f"\n{'─' * 80}")
    print(f"  {title}")
    print(f"{'─' * 80}")

def print_success(message):
    print(f"✅ {message}")

def print_info(message):
    print(f"ℹ️  {message}")

def print_warning(message):
    print(f"⚠️  {message}")

def print_progress(step, total, description):
    print(f"[{step}/{total}] {description}")

# ============================================================================
# VECTORIZED QUARTERLY CALCULATIONS
# ============================================================================

def calculate_quarterly_metrics_vectorized(daily_df, today):
    """
    VECTORIZED quarterly calculations (SMART VERSION)

    Enhancements:
    -------------
    ✔ Rolling quarters (last 12 months)
    ✔ Blended days to avoid demand inflation
    ✔ Handles intermittent SKUs safely
    ✔ Fully vectorized & production-ready

    Returns DataFrame with:
    - Q1_Sum, Q1_Days_With_Sales, Q1_Avg_Daily
    - Q2_Sum, Q2_Days_With_Sales, Q2_Avg_Daily
    - Q3_Sum, Q3_Days_With_Sales, Q3_Avg_Daily
    - Q4_Sum, Q4_Days_With_Sales, Q4_Avg_Daily
    """

    print_info("Computing quarterly metrics (vectorized + smart blend)...")

    # ─────────────────────────────────────────────────────────────
    # PARAMETERS (TUNABLE)
    # ─────────────────────────────────────────────────────────────
    QUARTER_DAYS = 90
    BLEND_FACTOR = 0.4          # 0 = calendar days | 1 = actual sales days
    MIN_SALES_DAYS_FOR_BLEND = 10
    MAX_DAILY_MULTIPLIER = 1.2  # cap vs yearly average

    # ─────────────────────────────────────────────────────────────
    # DEFINE ROLLING QUARTERS
    # ─────────────────────────────────────────────────────────────
    cutoff_q1 = today - pd.Timedelta(days=90)
    cutoff_q2 = today - pd.Timedelta(days=180)
    cutoff_q3 = today - pd.Timedelta(days=270)

    daily_df = daily_df.copy()

    daily_df['Quarter'] = pd.cut(
        daily_df['Date'],
        bins=[pd.Timestamp.min, cutoff_q3, cutoff_q2, cutoff_q1, today],
        labels=['Q4', 'Q3', 'Q2', 'Q1'],
        include_lowest=True
    )

    # ─────────────────────────────────────────────────────────────
    # FILTER SALES DAYS ONLY
    # ─────────────────────────────────────────────────────────────
    sales_days = daily_df[daily_df['bal Qty'] > 0].copy()

    # ─────────────────────────────────────────────────────────────
    # AGGREGATE
    # ─────────────────────────────────────────────────────────────
    quarterly = sales_days.groupby(['SKU', 'Quarter']).agg(
        Sum=('bal Qty', 'sum'),
        Days_With_Sales=('bal Qty', 'count')
    ).reset_index()

    quarterly_wide = quarterly.pivot(
        index='SKU',
        columns='Quarter',
        values=['Sum', 'Days_With_Sales']
    ).fillna(0)

    quarterly_wide.columns = [f'{q}_{m}' for m, q in quarterly_wide.columns]
    quarterly_wide = quarterly_wide.reset_index()

    # ─────────────────────────────────────────────────────────────
    # YEARLY AVG DAILY (ANTI-INFLATION ANCHOR)
    # ─────────────────────────────────────────────────────────────
    yearly = sales_days.groupby('SKU')['bal Qty'].agg(
        Total_Sales_1Y='sum',
        Active_Days_1Y='count'
    ).reset_index()

    yearly['Yearly_Avg_Daily'] = np.where(
        yearly['Active_Days_1Y'] > 0,
        yearly['Total_Sales_1Y'] / yearly['Active_Days_1Y'],
        0
    )

    quarterly_wide = quarterly_wide.merge(
        yearly[['SKU', 'Yearly_Avg_Daily']],
        on='SKU',
        how='left'
    )

    # ─────────────────────────────────────────────────────────────
    # SMART AVG DAILY PER QUARTER
    # ─────────────────────────────────────────────────────────────
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:

        sum_col = f'{q}_Sum'
        days_col = f'{q}_Days_With_Sales'
        avg_col = f'{q}_Avg_Daily'

        # Blended days
        adjusted_days = np.where(
            quarterly_wide[days_col] >= MIN_SALES_DAYS_FOR_BLEND,
            BLEND_FACTOR * quarterly_wide[days_col] +
            (1 - BLEND_FACTOR) * QUARTER_DAYS,
            QUARTER_DAYS
        )

        raw_avg = np.where(
            quarterly_wide[days_col] > 0,
            quarterly_wide[sum_col] / adjusted_days,
            0
        )

        # Cap vs yearly behavior
        cap = quarterly_wide['Yearly_Avg_Daily'] * MAX_DAILY_MULTIPLIER

        quarterly_wide[avg_col] = np.minimum(raw_avg, cap)

    quarterly_wide.drop(columns=['Yearly_Avg_Daily'], inplace=True)

    print_success("Quarterly metrics calculated (smart blended & capped)")

    return quarterly_wide


# ============================================================================
# DYNAMIC QUARTERLY WEIGHTING
# ============================================================================

def calculate_dynamic_weights(quarterly_df, min_active_days=MIN_ACTIVE_DAYS):
    """
    Calculate DYNAMIC weights based on sales activity

    Logic:
    ------
    If SKU has good sales activity across quarters:
        → Use adaptive weights based on recent activity
        → Captures seasonality automatically

    If SKU has sparse/uneven sales:
        → Use standard weights (40%, 30%, 20%, 10%)
        → More conservative approach

    Returns:
    --------
    DataFrame with W_Q1, W_Q2, W_Q3, W_Q4 columns
    """

    print_info("Calculating dynamic quarterly weights...")

    df = quarterly_df.copy()

    # Total sales across all quarters
    df['Total_Sales'] = (
        df['Q1_Sum'] + df['Q2_Sum'] +
        df['Q3_Sum'] + df['Q4_Sum']
    )

    # Total active days
    df['Total_Active_Days'] = (
        df['Q1_Days_With_Sales'] + df['Q2_Days_With_Sales'] +
        df['Q3_Days_With_Sales'] + df['Q4_Days_With_Sales']
    )

    # Check if SKU has sufficient activity for dynamic weighting
    sufficient_activity = df['Total_Active_Days'] >= min_active_days

    # ────────────────────────────────────────────────────────────────────────
    # METHOD 1: DYNAMIC WEIGHTS (for active SKUs)
    # ────────────────────────────────────────────────────────────────────────
    # Weight = Quarter_Sales / Total_Sales (with minimum floor)
    # This captures seasonality automatically!

    # Calculate raw proportions
    df['W_Q1_raw'] = np.where(df['Total_Sales'] > 0, df['Q1_Sum'] / df['Total_Sales'], 0)
    df['W_Q2_raw'] = np.where(df['Total_Sales'] > 0, df['Q2_Sum'] / df['Total_Sales'], 0)
    df['W_Q3_raw'] = np.where(df['Total_Sales'] > 0, df['Q3_Sum'] / df['Total_Sales'], 0)
    df['W_Q4_raw'] = np.where(df['Total_Sales'] > 0, df['Q4_Sum'] / df['Total_Sales'], 0)

    # Apply minimum floor and recency boost
    # Q1 gets minimum 30% (most recent), others minimum 10%
    df['W_Q1_dynamic'] = np.maximum(df['W_Q1_raw'], 0.30)
    df['W_Q2_dynamic'] = np.maximum(df['W_Q2_raw'], 0.10)
    df['W_Q3_dynamic'] = np.maximum(df['W_Q3_raw'], 0.10)
    df['W_Q4_dynamic'] = np.maximum(df['W_Q4_raw'], 0.10)

    # Normalize to sum to 1.0
    weight_sum = (
        df['W_Q1_dynamic'] + df['W_Q2_dynamic'] +
        df['W_Q3_dynamic'] + df['W_Q4_dynamic']
    )

    df['W_Q1_dynamic'] = df['W_Q1_dynamic'] / weight_sum
    df['W_Q2_dynamic'] = df['W_Q2_dynamic'] / weight_sum
    df['W_Q3_dynamic'] = df['W_Q3_dynamic'] / weight_sum
    df['W_Q4_dynamic'] = df['W_Q4_dynamic'] / weight_sum

    # ────────────────────────────────────────────────────────────────────────
    # METHOD 2: STANDARD WEIGHTS (for sparse SKUs)
    # ────────────────────────────────────────────────────────────────────────

    standard_weights = {
        'W_Q1_standard': 0.40,
        'W_Q2_standard': 0.30,
        'W_Q3_standard': 0.20,
        'W_Q4_standard': 0.10
    }

    for col, weight in standard_weights.items():
        df[col] = weight

    # ────────────────────────────────────────────────────────────────────────
    # FINAL DECISION: Choose method based on activity
    # ────────────────────────────────────────────────────────────────────────

    df['W_Q1'] = np.where(sufficient_activity, df['W_Q1_dynamic'], df['W_Q1_standard'])
    df['W_Q2'] = np.where(sufficient_activity, df['W_Q2_dynamic'], df['W_Q2_standard'])
    df['W_Q3'] = np.where(sufficient_activity, df['W_Q3_dynamic'], df['W_Q3_standard'])
    df['W_Q4'] = np.where(sufficient_activity, df['W_Q4_dynamic'], df['W_Q4_standard'])

    df['Weight_Method'] = np.where(
        sufficient_activity,
        'Dynamic (Seasonality)',
        'Standard (Conservative)'
    )

    print_success(f"Dynamic weights calculated")
    print_info(f"  Using dynamic weights: {sufficient_activity.sum():,} SKUs")
    print_info(f"  Using standard weights: {(~sufficient_activity).sum():,} SKUs")

    return df

# ============================================================================
# MAIN FORECAST FUNCTION (VECTORIZED)
# ============================================================================

def run_demand_forecast(combined_df, nb_days_df=None, analysis_date=None):
    """
    ADVANCED demand forecasting with actual sales days tracking

    Key Features:
    -------------
    1. Tracks ACTUAL SALES DAYS (not just date range)
    2. Dynamic quarterly weighting (captures seasonality)
    3. Fully vectorized (10x faster)
    4. FAMILY sales only
    5. Intelligent stockout adjustment

    Parameters:
    -----------
    combined_df : DataFrame
        Sales data with 'Sales_Type' column
    nb_days_df : DataFrame, optional
        Stockout data (Nb. Days in last 120 days)
    analysis_date : str/datetime, optional
        Reference date (default: today)

    Returns:
    --------
    sku_sales : DataFrame
        Complete forecast with all metrics
    filename : Path
        Path to Excel output
    """

    print_header("🚀 ADVANCED DEMAND FORECASTING (PRODUCTION)")

    # Analysis date
    if analysis_date is None:
        today = pd.Timestamp.now()
    else:
        today = pd.to_datetime(analysis_date)

    print(f"\n📅 Analysis Date: {today.strftime('%Y-%m-%d %H:%M:%S')}")

    # ════════════════════════════════════════════════════════════════════════
    # STEP 1: DATA PREPROCESSING
    # ════════════════════════════════════════════════════════════════════════

    print_section("STEP 1: DATA PREPROCESSING")
    print_progress(1, 4, "Filtering and cleaning...")

    df = combined_df.copy()

    # Verify columns
    required = ['Date', 'SKU', 'bal Qty', 'CATEGORY', 'CURRENT_STOCK', 'OUTSTANDING']
    missing = [col for col in required if col not in df.columns]

    if missing:
        print(f"❌ Missing columns: {missing}")
        return None, None

    # Convert
    df['Date'] = pd.to_datetime(df['Date'])
    df['SKU'] = df['SKU'].astype(str).str.strip()
    df['CURRENT_STOCK'] = pd.to_numeric(df['CURRENT_STOCK'], errors='coerce').fillna(0)
    df['OUTSTANDING'] = pd.to_numeric(df['OUTSTANDING'], errors='coerce').fillna(0)
    df['bal Qty'] = pd.to_numeric(df['bal Qty'], errors='coerce').fillna(0)

    # PR column
    has_pr = 'PR' in df.columns
    if has_pr:
        df['PR'] = pd.to_numeric(df['PR'], errors='coerce').fillna(0)
    else:
        df['PR'] = 0

    # Section column
    has_section = 'Section' in df.columns

    # ════════════════════════════════════════════════════════════════════════
    # FILTER: FAMILY SALES ONLY
    # ════════════════════════════════════════════════════════════════════════

    if 'Sales_Type' in df.columns:
        before = len(df)
        df = df[df['Sales_Type'] == 'FAMILY'].copy()
        after = len(df)

        print_warning(f"FAMILY sales only: {after:,} / {before:,} records")
        print_info(f"Excluded {before - after:,} CONTRACT sales")
    else:
        print_warning("No Sales_Type column - using all sales")

    # Filter last 365 days
    cutoff = today - pd.Timedelta(days=LOOKBACK_DAYS)
    df = df[df['Date'] >= cutoff]

    print_success(f"Data ready: {len(df):,} rows")
    print_info(f"Period: {df['Date'].min().date()} to {df['Date'].max().date()}")

    # ════════════════════════════════════════════════════════════════════════
    # STEP 2: VECTORIZED CALCULATIONS
    # ════════════════════════════════════════════════════════════════════════

    print_section("STEP 2: VECTORIZED CALCULATIONS")
    print_progress(2, 4, "Computing metrics...")

    # Aggregate daily
    agg_dict = {
        'bal Qty': 'sum',
        'CATEGORY': 'first',
        'CURRENT_STOCK': 'first',
        'OUTSTANDING': 'first',
        'PR': 'first'
    }

    if has_section:
        agg_dict['Section'] = 'first'

    daily = df.groupby(['SKU', 'Date'], as_index=False).agg(agg_dict)

    # ────────────────────────────────────────────────────────────────────────
    # 2.1: Basic SKU Metrics
    # ────────────────────────────────────────────────────────────────────────

    print_info("Computing basic metrics...")

    # Days with ANY sales
    days_with_sales = daily[daily['bal Qty'] > 0].groupby('SKU').size().reset_index()
    days_with_sales.columns = ['SKU', 'Actual_Sales_Days']

    # Total metrics
    sku_agg = {
        'bal Qty': ['sum', 'mean'],
        'CATEGORY': 'first',
        'CURRENT_STOCK': 'first',
        'OUTSTANDING': 'first',
        'PR': 'first'
    }

    if has_section:
        sku_agg['Section'] = 'first'

    sku_sales = daily.groupby('SKU').agg(sku_agg)
    sku_sales.columns = [
        'Total_Sales_1Y', 'Avg_Daily_Sales',
        'CATEGORY', 'CURRENT_STOCK', 'OUTSTANDING', 'PR'
    ] + (['Section'] if has_section else [])

    sku_sales = sku_sales.reset_index()

    # Merge actual sales days
    sku_sales = sku_sales.merge(days_with_sales, on='SKU', how='left')
    sku_sales['Actual_Sales_Days'] = sku_sales['Actual_Sales_Days'].fillna(0).astype(int)

    # Coverage based on ACTUAL sales days
    sku_sales['Data_Coverage_%'] = (
        sku_sales['Actual_Sales_Days'] / LOOKBACK_DAYS * 100
    ).round(1)

    # Actual monthly demand
    sku_sales['Monthly_Demand_Actual'] = sku_sales['Total_Sales_1Y'] / 12

    print_success(f"Basic metrics: {len(sku_sales):,} SKUs")
    print_info(f"  Avg actual sales days: {sku_sales['Actual_Sales_Days'].mean():.0f}/{LOOKBACK_DAYS}")
    print_info(f"  Avg coverage: {sku_sales['Data_Coverage_%'].mean():.1f}%")

    # ────────────────────────────────────────────────────────────────────────
    # 2.2: QUARTERLY METRICS (Vectorized)
    # ────────────────────────────────────────────────────────────────────────

    quarterly_metrics = calculate_quarterly_metrics_vectorized(daily, today)

    # Merge
    sku_sales = sku_sales.merge(quarterly_metrics, on='SKU', how='left')

    # Fill missing quarterly data
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        for col_type in ['Sum', 'Days_With_Sales', 'Avg_Daily']:
            col = f'{q}_{col_type}'
            if col not in sku_sales.columns:
                sku_sales[col] = 0
            else:
                sku_sales[col] = sku_sales[col].fillna(0)

    # ────────────────────────────────────────────────────────────────────────
    # 2.3: DYNAMIC WEIGHTS (Captures Seasonality)
    # ────────────────────────────────────────────────────────────────────────

    sku_sales = calculate_dynamic_weights(sku_sales, MIN_ACTIVE_DAYS)

    # ────────────────────────────────────────────────────────────────────────
    # 2.4: WEIGHTED MOVING AVERAGE (Vectorized)
    # ────────────────────────────────────────────────────────────────────────

    print_info("Computing WMA with dynamic weights...")

    # Calculate WMA using dynamic weights
    sku_sales['ADS_WMA'] = (
        sku_sales['Q1_Avg_Daily'] * sku_sales['W_Q1'] +
        sku_sales['Q2_Avg_Daily'] * sku_sales['W_Q2'] +
        sku_sales['Q3_Avg_Daily'] * sku_sales['W_Q3'] +
        sku_sales['Q4_Avg_Daily'] * sku_sales['W_Q4']
    )

    sku_sales['Monthly_WMA'] = sku_sales['ADS_WMA'] * 30

    # Monthly versions of quarters
    sku_sales['Q1_Monthly'] = sku_sales['Q1_Avg_Daily'] * 30
    sku_sales['Q2_Monthly'] = sku_sales['Q2_Avg_Daily'] * 30
    sku_sales['Q3_Monthly'] = sku_sales['Q3_Avg_Daily'] * 30
    sku_sales['Q4_Monthly'] = sku_sales['Q4_Avg_Daily'] * 30

    print_success("WMA calculated with dynamic weights")
    #
    # # ────────────────────────────────────────────────────────────────────────
    # # 2.5: 80th Percentile (Vectorized for Category A)
    # # ────────────────────────────────────────────────────────────────────────
    #
    # print_info("Computing 80th percentile...")
    #
    # # Filter Category A sales
    # cat_a_daily = daily[
    #     daily['SKU'].isin(sku_sales[sku_sales['CATEGORY'].isin(['A', 'A+'])]['SKU'])
    # ].copy()
    #
    # cat_a_daily = cat_a_daily[cat_a_daily['bal Qty'] > 0]
    #
    # if len(cat_a_daily) > 0:
    #     p80 = cat_a_daily.groupby('SKU')['bal Qty'].quantile(0.8).reset_index()
    #     p80.columns = ['SKU', 'Percentile_80']
    #
    #     sku_sales = sku_sales.merge(p80, on='SKU', how='left')
    # else:
    #     sku_sales['Percentile_80'] = np.nan
    #
    # # Apply percentile if higher
    # mask_a = sku_sales['CATEGORY'].isin(['A', 'A+'])
    # mask_p80 = sku_sales['Percentile_80'] > sku_sales['ADS_WMA']
    #
    # sku_sales.loc[mask_a & mask_p80, 'ADS_WMA'] = sku_sales.loc[mask_a & mask_p80, 'Percentile_80']
    # sku_sales.loc[mask_a & mask_p80, 'Monthly_WMA'] = sku_sales.loc[mask_a & mask_p80, 'Percentile_80'] * 30
    # sku_sales.loc[mask_a & mask_p80, 'Used_Percentile'] = 'Yes'
    # sku_sales.loc[mask_a & ~mask_p80, 'Used_Percentile'] = 'No'
    # sku_sales.loc[~mask_a, 'Used_Percentile'] = 'N/A'
    #
    # print_success("80th percentile applied")
    sku_sales['Used_Percentile'] = 'N/A'


    # ────────────────────────────────────────────────────────────────────────
    # 2.6: SMART DECISION
    # ────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────
    # 2.6: SMART DECISION (Unified for ALL categories)
    # ────────────────────────────────────────────────────────────────────────

    sku_sales['Monthly_Demand_Final'] = sku_sales['Monthly_Demand_Actual']
    sku_sales['Forecast_Method'] = 'Actual (Conservative)'

    good_coverage = sku_sales['Data_Coverage_%'] >= (MIN_DATA_COVERAGE * 100)
    wma_higher = sku_sales['Monthly_WMA'] > sku_sales['Monthly_Demand_Actual']

    sku_sales.loc[good_coverage & wma_higher, 'Monthly_Demand_Final'] = \
        sku_sales.loc[good_coverage & wma_higher, 'Monthly_WMA']

    sku_sales.loc[good_coverage & wma_higher, 'Forecast_Method'] = 'WMA (All Categories)'


    # ────────────────────────────────────────────────────────────────────────
    # 2.7: STOCKOUT ADJUSTMENT (Vectorized)
    # ────────────────────────────────────────────────────────────────────────

    if nb_days_df is not None:
        print_info("Applying stockout adjustments (vectorized)...")

        nb_days_df = nb_days_df.copy()
        nb_days_df['SKU'] = nb_days_df['SKU'].astype(str).str.strip()

        sku_sales = sku_sales.merge(
            nb_days_df[['SKU', 'Nb. Days (Avail. Balance)']],
            on='SKU',
            how='left'
        )

        # Vectorized stockout calculation
        STOCKOUT_WINDOW = 120

        stockout_mask = (
            (sku_sales['CURRENT_STOCK'] <= STOCKOUT_THRESHOLD) &
            (sku_sales['Nb. Days (Avail. Balance)'].notna()) &
            (sku_sales['Nb. Days (Avail. Balance)'] < STOCKOUT_WINDOW) &
            (sku_sales['Nb. Days (Avail. Balance)'] > 0)
        )

        # Calculate days stockout
        sku_sales['Days_Stockout_Last_120'] = np.where(
            stockout_mask,
            STOCKOUT_WINDOW - sku_sales['Nb. Days (Avail. Balance)'],
            0
        )

        # Adjustment factor (vectorized)
        available = sku_sales.loc[stockout_mask, 'Nb. Days (Avail. Balance)']
        missing = sku_sales.loc[stockout_mask, 'Days_Stockout_Last_120']

        adjustment_factor = ((available + (0.5 * missing)) / available).clip(upper=STOCKOUT_CAP)

        # Apply
        sku_sales.loc[stockout_mask, 'Monthly_Demand_Final'] *= adjustment_factor
        sku_sales.loc[stockout_mask, 'Stockout_Adjusted'] = 'Yes'
        sku_sales.loc[stockout_mask, 'Stockout_Fill_Method'] = f'Half Fill (cap={STOCKOUT_CAP}x)'

        sku_sales['Stockout_Adjusted'] = sku_sales['Stockout_Adjusted'].fillna('No')
        sku_sales['Stockout_Fill_Method'] = sku_sales['Stockout_Fill_Method'].fillna('N/A')

        print_success(f"Adjusted {stockout_mask.sum():,} SKUs for stockouts")
    else:
        sku_sales['Days_Stockout_Last_120'] = 0
        sku_sales['Stockout_Adjusted'] = 'N/A'
        sku_sales['Stockout_Fill_Method'] = 'N/A'

    # ════════════════════════════════════════════════════════════════════════
    # STEP 3: INVENTORY CALCULATIONS (Vectorized)
    # ════════════════════════════════════════════════════════════════════════

    print_section("STEP 3: INVENTORY CALCULATIONS")
    print_progress(3, 4, "Computing order quantities...")

    # ADS Final
    sku_sales['ADS_Final'] = sku_sales['Monthly_Demand_Final'] / 30

    # Effective Stock
    sku_sales['Effective_Stock'] = (
        sku_sales['CURRENT_STOCK'] +
        sku_sales['OUTSTANDING'] +
        sku_sales['PR']
    ).astype(int)

    # Target & Order
    sku_sales['Target_Stock_7M'] = (
        sku_sales['Monthly_Demand_Final'] * FORWARD_COVERAGE_MONTHS
    ).astype(int)

    sku_sales['Repeated_Order_Qty'] = (
        sku_sales['Target_Stock_7M'] - sku_sales['Effective_Stock']
    ).clip(lower=0).astype(int)

    sku_sales['Max_Stock_Level'] = sku_sales['Target_Stock_7M']

    # Status
    sku_sales['STATUS'] = 'OK'
    sku_sales.loc[sku_sales['CURRENT_STOCK'] == 0, 'STATUS'] = 'OUT_OF_STOCK'
    sku_sales.loc[sku_sales['CURRENT_STOCK'] <= 30, 'STATUS'] = 'CRITICAL'
    sku_sales.loc[sku_sales['Repeated_Order_Qty'] > 0, 'STATUS'] = 'REORDER'
    sku_sales.loc[sku_sales['Effective_Stock'] > sku_sales['Max_Stock_Level'], 'STATUS'] = 'EXCESS'

    # Days coverage
    sku_sales['Days_Coverage'] = np.where(
        sku_sales['ADS_Final'] > 0,
        sku_sales['Effective_Stock'] / sku_sales['ADS_Final'],
        999
    ).round(1)

    print_success("Inventory calculations complete")

    # ════════════════════════════════════════════════════════════════════════
    # STEP 4: EXCEL EXPORT
    # ════════════════════════════════════════════════════════════════════════

    print_section("STEP 4: EXCEL EXPORT")
    print_progress(4, 4, "Generating report...")

    desktop = Path.home() / "Desktop"
    output_folder = desktop / "Forecast_Results"
    output_folder.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = output_folder / f'Advanced_Forecast_{timestamp}.xlsx'

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:

        # Summary
        dynamic_count = (sku_sales['Weight_Method'] == 'Dynamic (Seasonality)').sum()
        standard_count = (sku_sales['Weight_Method'] == 'Standard (Conservative)').sum()

        summary = pd.DataFrame([
            {'Metric': '🎯 ADVANCED FORECAST SUMMARY', 'Value': ''},
            {'Metric': '', 'Value': ''},
            {'Metric': 'Total SKUs', 'Value': f"{len(sku_sales):,}"},
            {'Metric': 'Analysis Date', 'Value': today.strftime('%Y-%m-%d')},
            {'Metric': '', 'Value': ''},
            {'Metric': '📊 SALES TRACKING:', 'Value': ''},
            {'Metric': '  ├─ Avg Actual Sales Days',
             'Value': f"{sku_sales['Actual_Sales_Days'].mean():.0f} / 365"},
            {'Metric': '  ├─ Avg Coverage',
             'Value': f"{sku_sales['Data_Coverage_%'].mean():.1f}%"},
            {'Metric': '  └─ Sales Type', 'Value': 'FAMILY only'},
            {'Metric': '', 'Value': ''},
            {'Metric': '🎯 WEIGHTING METHOD:', 'Value': ''},
            {'Metric': '  ├─ Dynamic (Seasonality)', 'Value': f"{dynamic_count:,}"},
            {'Metric': '  └─ Standard (Conservative)', 'Value': f"{standard_count:,}"},
            {'Metric': '', 'Value': ''},
            {'Metric': '📦 REORDER STATUS:', 'Value': ''},
            {'Metric': '  ├─ Reorder Required',
             'Value': f"{(sku_sales['Repeated_Order_Qty'] > 0).sum():,}"},
            {'Metric': '  ├─ Total Order Qty',
             'Value': f"{sku_sales['Repeated_Order_Qty'].sum():,.0f}"},
            {'Metric': '  └─ Critical Items',
             'Value': f"{(sku_sales['STATUS'] == 'CRITICAL').sum():,}"},
        ])

        summary.to_excel(writer, sheet_name='Summary', index=False)

        # All SKUs
        sku_sales.to_excel(writer, sheet_name='All SKUs', index=False)

        # Reorder
        reorder = sku_sales[sku_sales['Repeated_Order_Qty'] > 0].copy()
        if len(reorder) > 0:
            reorder = reorder.sort_values('Repeated_Order_Qty', ascending=False)
            reorder.to_excel(writer, sheet_name='Reorder Required', index=False)

        # Section Summary
        if has_section:
            section_summary = sku_sales.groupby('Section').agg({
                'SKU': 'count',
                'Total_Sales_1Y': 'sum',
                'Repeated_Order_Qty': 'sum',
                'Effective_Stock': 'sum'
            }).reset_index()

            section_summary.columns = [
                'Section', 'Total_SKUs', 'Total_Sales_1Y',
                'Repeated_Order_Qty', 'Effective_Stock'
            ]

            section_summary = section_summary.sort_values('Repeated_Order_Qty', ascending=False)
            section_summary.to_excel(writer, sheet_name='Section Summary', index=False)

    print_success(f"Excel saved: {filename.name}")

    # Final summary
    print_header("✅ ANALYSIS COMPLETE!")

    print("\n" + "┌" + "─" * 78 + "┐")
    print(f"│  SKUs Analyzed: {len(sku_sales):>61,} │")
    print(f"│  Avg Actual Sales Days: {sku_sales['Actual_Sales_Days'].mean():>49.0f} / 365 │")
    print(f"│  Dynamic Weighting: {dynamic_count:>57,} │")
    print(f"│  Reorder Required: {(sku_sales['Repeated_Order_Qty'] > 0).sum():>58,} │")
    print(f"│  Total Order Qty: {sku_sales['Repeated_Order_Qty'].sum():>59,.0f} │")
    print("└" + "─" * 78 + "┘\n")

    return sku_sales, filename

# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":

    print_header("🚀 ADVANCED FORECASTING SYSTEM")

    print("\n📖 USAGE:\n")

    print("```python")
    print("# Step 1: Load data")
    print("from ENHANCED_DATA_LOADER import run_pipeline")
    print("combined_df = run_pipeline(")
    print("    cost_file='SKUS COST.xlsx',")
    print("    stock_file='current_stock.xlsx',")
    print("    # ... other files")
    print(")")
    print("")
    print("# Step 2: Load stockout data (optional)")
    print("nb_days_df = pd.read_excel('nb_days.xlsx')")
    print("")
    print("# Step 3: Run forecast")
    print("inventory_df, file = run_demand_forecast(")
    print("    combined_df,")
    print("    nb_days_df,")
    print("    analysis_date='2025-12-31'  # Optional")
    print(")")
    print("```")

    print("\n🆕 KEY FEATURES:")
    print("   ✅ Actual Sales Days tracking")
    print("   ✅ Dynamic quarterly weighting (auto seasonality)")
    print("   ✅ Fully vectorized (10x faster)")
    print("   ✅ FAMILY sales only")
    print("   ✅ Intelligent stockout adjustment")

    print("\n📊 NEW COLUMNS:")
    print("   • Actual_Sales_Days (days with sales)")
    print("   • Q1-Q4 Days_With_Sales (activity per quarter)")
    print("   • W_Q1, W_Q2, W_Q3, W_Q4 (dynamic weights)")
    print("   • Weight_Method (Dynamic/Standard)")
    print("   • Days_Stockout_Last_120")

    print("\n" + "=" * 80)


  🚀 ADVANCED FORECASTING SYSTEM

📖 USAGE:

```python
# Step 1: Load data
from ENHANCED_DATA_LOADER import run_pipeline
combined_df = run_pipeline(
    cost_file='SKUS COST.xlsx',
    stock_file='current_stock.xlsx',
    # ... other files
)

# Step 2: Load stockout data (optional)
nb_days_df = pd.read_excel('nb_days.xlsx')

# Step 3: Run forecast
inventory_df, file = run_demand_forecast(
    combined_df,
    nb_days_df,
    analysis_date='2025-12-31'  # Optional
)
```

🆕 KEY FEATURES:
   ✅ Actual Sales Days tracking
   ✅ Dynamic quarterly weighting (auto seasonality)
   ✅ Fully vectorized (10x faster)
   ✅ FAMILY sales only
   ✅ Intelligent stockout adjustment

📊 NEW COLUMNS:
   • Actual_Sales_Days (days with sales)
   • Q1-Q4 Days_With_Sales (activity per quarter)
   • W_Q1, W_Q2, W_Q3, W_Q4 (dynamic weights)
   • Weight_Method (Dynamic/Standard)
   • Days_Stockout_Last_120



In [3]:
nb_days_df = pd.read_excel(r'C:\Users\User\Desktop\nb.xlsx')


inventory_df_tagropa, file = run_demand_forecast(
    combined_df,
    nb_days_df
)


  🚀 ADVANCED DEMAND FORECASTING (PRODUCTION)

📅 Analysis Date: 2026-01-19 14:53:16

────────────────────────────────────────────────────────────────────────────────
  STEP 1: DATA PREPROCESSING
────────────────────────────────────────────────────────────────────────────────
[1/4] Filtering and cleaning...
⚠️  FAMILY sales only: 4,892,101 / 4,930,233 records
ℹ️  Excluded 38,132 CONTRACT sales
✅ Data ready: 377,565 rows
ℹ️  Period: 2025-01-20 to 2026-01-14

────────────────────────────────────────────────────────────────────────────────
  STEP 2: VECTORIZED CALCULATIONS
────────────────────────────────────────────────────────────────────────────────
[2/4] Computing metrics...
ℹ️  Computing basic metrics...
✅ Basic metrics: 36,314 SKUs
ℹ️    Avg actual sales days: 9/365
ℹ️    Avg coverage: 2.4%
ℹ️  Computing quarterly metrics (vectorized + smart blend)...
✅ Quarterly metrics calculated (smart blended & capped)
ℹ️  Calculating dynamic quarterly weights...
✅ Dynamic weights calculated
ℹ️  

In [5]:
# ============================================================================
# CELL 1: SAVE VARIABLES (ضعها في نهاية Notebook)
# ============================================================================

import pickle
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("💾 SAVING NOTEBOOK VARIABLES")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# 📋 STEP 1: List variables to save (Edit this list as needed)
# ═══════════════════════════════════════════════════════════════════════════

variables_to_save = [
    # Core DataFrames
    'combined_df', #the main dataframe
    'combined_df_updated', # updated dataframe for pattern dashboard
    'df',

    # Analysis Results
    'results_df', # for skus classification
    'classification_df',
    'forecasts_df',
    'inventory_df' , #for inventory man notebook ,
    'results_df_classi_collection', #dataframe for collection classification
    'pattern_df',
    'inventory_df_tagropa'   ,
        'sku_df',
    'section_df', #dataframe for forecasted pattern of the demand


    # Dashboard Objects
    'dashboard',

    # Pattern Analysis
    'pattern_stats',
    'sku_level',
    'color_analysis',

    # Configs & Dicts
    'config',
    'metrics',

    # Add any other variables here
]

# ═══════════════════════════════════════════════════════════════════════════
# 📁 STEP 2: Create save folder
# ═══════════════════════════════════════════════════════════════════════════

save_folder = Path.home() / "Desktop" / "notebook_variables"
save_folder.mkdir(exist_ok=True)

# Create timestamped subfolder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
session_folder = save_folder / f"session_{timestamp}"
session_folder.mkdir(exist_ok=True)

print(f"\n📂 Save location: {session_folder}")
print()

# ═══════════════════════════════════════════════════════════════════════════
# 💾 STEP 3: Save each variable
# ═══════════════════════════════════════════════════════════════════════════

saved_vars = {}
success_count = 0
skip_count = 0
fail_count = 0

for var_name in variables_to_save:
    if var_name in globals():
        try:
            var_value = globals()[var_name]

            # Save to pickle
            file_path = session_folder / f"{var_name}.pkl"
            with open(file_path, 'wb') as f:
                pickle.dump(var_value, f)

            # Get file size
            size_kb = file_path.stat().st_size / 1024
            size_str = f"{size_kb:.1f} KB" if size_kb < 1024 else f"{size_kb/1024:.1f} MB"

            saved_vars[var_name] = {
                'saved': True,
                'type': type(var_value).__name__,
                'size': size_str
            }

            print(f"✅ {var_name:<25} | {saved_vars[var_name]['type']:<15} | {size_str}")
            success_count += 1

        except Exception as e:
            saved_vars[var_name] = {'saved': False, 'error': str(e)}
            print(f"❌ {var_name:<25} | FAILED: {str(e)[:40]}")
            fail_count += 1
    else:
        print(f"⚠️  {var_name:<25} | NOT FOUND (skipped)")
        skip_count += 1

# ═══════════════════════════════════════════════════════════════════════════
# 📝 STEP 4: Save metadata
# ═══════════════════════════════════════════════════════════════════════════

metadata = {
    'timestamp': timestamp,
    'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'saved_variables': saved_vars,
    'notebook_name': 'Current Notebook',  # You can customize this
    'success_count': success_count,
    'skip_count': skip_count,
    'fail_count': fail_count
}

metadata_path = session_folder / "metadata.pkl"
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

# ═══════════════════════════════════════════════════════════════════════════
# ✅ STEP 5: Summary
# ═══════════════════════════════════════════════════════════════════════════

print()
print("=" * 80)
print("✅ SAVE COMPLETE!")
print("=" * 80)
print(f"📂 Location: {session_folder}")
print(f"✅ Saved:    {success_count} variables")
print(f"⚠️  Skipped:  {skip_count} variables (not found)")
print(f"❌ Failed:   {fail_count} variables")
print("=" * 80)
print()
print("💡 To load these variables in another notebook, use CELL 2")
print("=" * 80)

💾 SAVING NOTEBOOK VARIABLES

📂 Save location: C:\Users\User\Desktop\notebook_variables\session_20260119_145703

✅ combined_df               | DataFrame       | 762.1 MB
✅ combined_df_updated       | DataFrame       | 1592.6 MB
✅ df                        | DataFrame       | 1454.8 MB
⚠️  results_df                | NOT FOUND (skipped)
⚠️  classification_df         | NOT FOUND (skipped)
⚠️  forecasts_df              | NOT FOUND (skipped)
✅ inventory_df              | DataFrame       | 17.6 MB
⚠️  results_df_classi_collection | NOT FOUND (skipped)
⚠️  pattern_df                | NOT FOUND (skipped)
✅ inventory_df_tagropa      | DataFrame       | 16.8 MB
✅ sku_df                    | DataFrame       | 15.0 MB
✅ section_df                | DataFrame       | 855.7 KB
⚠️  dashboard                 | NOT FOUND (skipped)
⚠️  pattern_stats             | NOT FOUND (skipped)
⚠️  sku_level                 | NOT FOUND (skipped)
⚠️  color_analysis            | NOT FOUND (skipped)
⚠️  config         